In [ ]:
!pip install ortools

from ortools.constraint_solver import pywrapcp, routing_enums_pb2
import numpy as np


In [ ]:
# Danh sách điểm đúng thứ tự như trong Excel
locations = ["W","LAT","UoB","LRP","CGB","LDHS","LTC","HWI","CDH","BBP"]

# Mapping tên -> index
idx = {name: i for i, name in enumerate(locations)}

# Ma trận khoảng cách (km) – copy đúng từ Excel
dist = np.array([
    [0,   4.2, 2.5, 2.5, 3.1, 4.0, 2.0, 3.5, 4.1, 6.0],
    [4.2, 0,   3.8, 3.8, 2.0, 5.5, 4.5, 6.0, 6.5, 7.2],
    [2.5, 3.8, 0,   2.0, 2.5, 3.0, 1.0, 3.2, 3.8, 5.5],
    [2.5, 3.8, 2.0, 0,   2.5, 3.0, 2.2, 3.2, 4.0, 5.5],
    [3.1, 2.0, 2.5, 2.5, 0,   4.0, 2.2, 3.5, 5.0, 6.8],
    [4.0, 5.5, 3.0, 3.0, 4.0, 0,   2.5, 2.0, 2.8, 5.0],
    [2.0, 4.5, 1.0, 2.2, 2.2, 2.5, 0,   2.8, 3.5, 5.2],
    [3.5, 6.0, 3.2, 3.2, 3.5, 2.0, 2.8, 0,   1.5, 4.8],
    [4.1, 6.5, 3.8, 4.0, 5.0, 2.8, 3.5, 1.5, 0,   4.5],
    [6.0, 7.2, 5.5, 5.5, 6.8, 5.0, 5.2, 4.8, 4.5, 0  ]
])

# Các cụm theo phương án 4 vans trong Excel
clusters = [
    ["UoB", "LTC"],            # Van 1
    ["LDHS", "HWI"],           # Van 2
    ["LAT", "CGB"],            # Van 3
    ["LRP", "CDH", "BBP"]      # Van 4
]


In [ ]:
def solve_tsp_for_cluster(cluster_nodes):
    """
    cluster_nodes: list tên điểm, VD ["UoB","LTC"]
    Giải TSP: W -> cluster nodes -> W
    """
    # Subset các node: luôn có depot W + các node trong cụm
    sub_locs = ["W"] + cluster_nodes
    n = len(sub_locs)

    # Map sub_index -> global index
    sub_to_global = [idx[name] for name in sub_locs]

    # Build Routing model với 1 vehicle
    manager = pywrapcp.RoutingIndexManager(n, 1, 0)  # depot = 0 (W trong subset)
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        g_from = sub_to_global[manager.IndexToNode(from_index)]
        g_to   = sub_to_global[manager.IndexToNode(to_index)]
        return int(dist[g_from][g_to] * 1000)  # dùng số nguyên

    transit_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_index)

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    search_parameters.time_limit.FromSeconds(5)

    solution = routing.SolveWithParameters(search_parameters)

    # Extract route
    index = routing.Start(0)
    route_names = []
    route_distance = 0.0

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        route_names.append(sub_locs[node])
        next_index = solution.Value(routing.NextVar(index))
        if not routing.IsEnd(next_index):
            g_from = sub_to_global[node]
            g_to   = sub_to_global[manager.IndexToNode(next_index)]
            route_distance += dist[g_from][g_to]
        index = next_index

    route_names.append("W")  # quay lại kho

    return route_names, route_distance

# Chạy cho 4 vans
total = 0.0
all_routes = []

for van_id, cl in enumerate(clusters, start=1):
    route, d = solve_tsp_for_cluster(cl)
    all_routes.append((van_id, route, d))
    total += d

total, all_routes


(np.float64(26.1),
 [(1, ['W', 'LTC', 'UoB', 'W'], np.float64(3.0)),
  (2, ['W', 'HWI', 'LDHS', 'W'], np.float64(5.5)),
  (3, ['W', 'CGB', 'LAT', 'W'], np.float64(5.1)),
  (4, ['W', 'LRP', 'BBP', 'CDH', 'W'], np.float64(12.5))])